# Custom Tasks Tutorial

This tutorial shows you how to create custom segmentation tasks with nnunetsegmentator.

## Why Custom Tasks?

Custom tasks allow you to:
- Use your own trained nnUNet models
- Define custom preprocessing and postprocessing pipelines
- Configure specific parameters for your use case
- Manage multiple models in a single application

## Basic Custom Task

### Step 1: Define Your Task Configuration

In [ ]:
from nnunetsegmentator import TaskRegistry, Config
from nnunetsegmentator.pipeline.base import BasePreprocessor, BasePostprocessor

# Define custom preprocessor
class MyPreprocessor(BasePreprocessor):
    def preprocess(self, image):
        # Your custom preprocessing logic
        # Example: Normalize to specific range
        image_array = image.numpy()
        image_array = (image_array - image_array.min()) / (image_array.max() - image_array.min() + 1e-8)
        return image_array

# Define custom postprocessor
class MyPostprocessor(BasePostprocessor):
    def postprocess(self, segmentation):
        # Your custom postprocessing logic
        # Example: Remove small components
        # Your custom logic here
        return segmentation

# Register your task
TaskRegistry.register_task(
    task_name='my_custom_task',
    model_path='/path/to/your/nnunet/model',
    config=Config(
        use_gpu=True,
        num_workers=4,
        batch_size=4
    ),
    preprocessor=MyPreprocessor(),
    postprocessor=MyPostprocessor(),
    labels={
        'background': 0,
        'structure1': 1,
        'structure2': 2
    }
)


### Step 2: Use Your Custom Task

In [ ]:
from nnunetsegmentator import SegmentationOrchestrator

# Initialize with your custom task
orchestrator = SegmentationOrchestrator(
    task_name='my_custom_task'
)

# Segment using your custom task
result = orchestrator.segment(
    input_data='path/to/image.nii.gz',
    output_path='path/to/output.nii.gz',
    return_labels=True,
    compute_metrics=True
)

print(f"Segmentation complete!")
print(f"Labels: {list(result.labels.keys())}")

## Advanced Custom Task

### Multiple Models Pipeline

Use multiple models for different structures:

In [ ]:
from nnunetsegmentator import TaskRegistry, Config, SegmentationOrchestrator

# Register multiple tasks for different structures
TaskRegistry.register_task(
    task_name='task_structure_a',
    model_path='/path/to/model_structure_a',
    config=Config(use_gpu=True, batch_size=4),
    labels={'background': 0, 'structure_a': 1}
)

TaskRegistry.register_task(
    task_name='task_structure_b',
    model_path='/path/to/model_structure_b',
    config=Config(use_gpu=True, batch_size=4),
    labels={'background': 0, 'structure_b': 1}
)

# Create orchestrator for structure A
orchestrator_a = SegmentationOrchestrator(
    task_name='task_structure_a'
)

# Create orchestrator for structure B
orchestrator_b = SegmentationOrchestrator(
    task_name='task_structure_b'
)

# Segment both structures
result_a = orchestrator_a.segment(
    input_data='path/to/image.nii.gz',
    output_path='path/to/structure_a.nii.gz'
)

result_b = orchestrator_b.segment(
    input_data='path/to/image.nii.gz',
    output_path='path/to/structure_b.nii.gz'
)

print(f"Structure A volume: {result_a.metrics['volume_mm3']:.2f} mm³")
print(f"Structure B volume: {result_b.metrics['volume_mm3']:.2f} mm³")

### Custom Pipeline with Multiple Steps

In [ ]:
from nnunetsegmentator.pipeline.base import BasePreprocessor, BasePostprocessor
import numpy as np

class MultiStepPreprocessor(BasePreprocessor):
    def __init__(self, steps=None):
        self.steps = steps or []
    
    def add_step(self, step_func):
        self.steps.append(step_func)
    
    def preprocess(self, image):
        image_array = image.numpy()
        
        # Apply each preprocessing step
        for step in self.steps:
            image_array = step(image_array)
        
        return image_array

# Create custom preprocessor
preprocessor = MultiStepPreprocessor()

# Add preprocessing steps
def normalize(image):
    return (image - image.mean()) / (image.std() + 1e-8)

def clip(image, lower=0.01, upper=0.99):
    p_lower = np.percentile(image, lower * 100)
    p_upper = np.percentile(image, upper * 100)
    return np.clip(image, p_lower, p_upper)

preprocessor.add_step(clip)
preprocessor.add_step(normalize)

# Register task with custom preprocessor
TaskRegistry.register_task(
    task_name='multi_step_task',
    model_path='/path/to/model',
    config=Config(use_gpu=True),
    preprocessor=preprocessor,
    labels={'background': 0, 'structure': 1}
)

## Model Management

### Loading Models from Different Sources

In [ ]:
from nnunetsegmentator import TaskRegistry, Config, SegmentationOrchestrator
from pathlib import Path

# Option 1: Local model path
TaskRegistry.register_task(
    task_name='local_model',
    model_path='/absolute/path/to/model',
    config=Config(use_gpu=True)
)

# Option 2: Model in nnUNet_results directory
TaskRegistry.register_task(
    task_name='nnunet_model',
    model_path='nnunet_results/Task001_TaskName/nnUNetTrainer__nnUNetPlans__config',
    config=Config(use_gpu=True)
)

# Option 3: Override model path at runtime
orchestrator = SegmentationOrchestrator(
    task_name='local_model',
    model_path='/different/path/to/model'  # Override
)

# Option 4: Load from configuration file
import yaml

with open('task_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

TaskRegistry.register_task(
    task_name='yaml_config_task',
    **config
)

### Task Registry Operations

In [ ]:
from nnunetsegmentator import TaskRegistry

# List all registered tasks
tasks = TaskRegistry.list_tasks()
print("Registered tasks:")
for task in tasks:
    print(f"  - {task}")

# Get task information
task_info = TaskRegistry.get_task_info('my_custom_task')
print(f"\nTask info: {task_info}")

# Check if task exists
if TaskRegistry.has_task('my_custom_task'):
    print("\nTask exists!")

# Unregister a task
TaskRegistry.unregister_task('my_custom_task')
print("\nTask unregistered")

## Best Practices

### 1. Organize Your Tasks

Create a task configuration file:

In [ ]:
import yaml
from pathlib import Path

# Create task_config.yaml
task_config = {
    'task_name': 'my_custom_task',
    'model_path': '/path/to/model',
    'config': {
        'use_gpu': True,
        'num_workers': 4,
        'batch_size': 4,
        'patch_size': [128, 128, 64]
    },
    'labels': {
        'background': 0,
        'structure1': 1,
        'structure2': 2
    }
}

Path('task_config.yaml').write_text(yaml.dump(task_config, default_flow_style=False))

# Load and register task
with open('task_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

TaskRegistry.register_task(**config)

### 2. Version Control Your Models

Track model versions in your task configuration:

In [ ]:
import yaml
from datetime import datetime

task_config = {
    'task_name': 'my_custom_task',
    'model_path': '/path/to/model',
    'model_version': 'v1.0.0',
    'training_date': '2024-01-15',
    'config': {
        'use_gpu': True,
        'num_workers': 4
    },
    'labels': {
        'background': 0,
        'structure': 1
    }
}

print(f"Task registered with model version: {task_config['model_version']}")
print(f"Training date: {task_config['training_date']}")

### 3. Test Your Custom Task

Always test your custom task before production use:

In [ ]:
from nnunetsegmentator import TaskRegistry, SegmentationOrchestrator

# Test the task
try:
    orchestrator = SegmentationOrchestrator(
        task_name='my_custom_task'
    )
    
    # Test with a sample image
    result = orchestrator.segment(
        input_data='path/to/test/image.nii.gz',
        output_path='path/to/test/output.nii.gz',
        return_labels=True,
        compute_metrics=True
    )
    
    print("✓ Task test successful!")
    print(f"  Labels: {list(result.labels.keys())}")
    print(f"  Output shape: {result.segmentation.GetSize()}")
    
    if result.metrics:
        print(f"  Volume: {result.metrics['volume_mm3']:.2f} mm³")
        
except Exception as e:
    print(f"✗ Task test failed: {e}")

## Next Steps

- Review [Quick Start](quick_start.ipynb) for basic usage
- Check [Advanced](advanced.ipynb) for performance optimization
- Explore [API Reference](../reference/api-reference.md) for detailed documentation